In [0]:
from pyspark.sql import functions as F

# Agrega receita total, ticket médio, taxa de ocupação e ROI estimado

silver_orders  = spark.table("silver.ticket_orders")
silver_events  = spark.table("silver.events")
silver_budgets = spark.table("silver.budgets")
silver_expenses = spark.table("silver.expenses")

# Receita de ingressos por evento
ticket_revenue = (
    silver_orders
    .filter(F.col("status") == "confirmed")
    .groupBy("event_id")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("total_paid").alias("gross_ticket_revenue_cents"),
        F.avg("total_paid").alias("avg_ticket_price_cents"),
    )
)

# Custo real por evento (soma de expenses)
event_costs = (
    spark.table("bronze.expenses").alias("exp")
    .join(spark.table("bronze.budgets").alias("b"),
          F.col("exp.budget_id") == F.col("b.id"))
    .groupBy("b.event_id")
    .agg(F.sum("exp.amount").alias("total_cost_cents"))
)

fact_event_revenue = (
    silver_events.alias("e")
    .join(ticket_revenue.alias("tr"),
          F.col("e.event_id") == F.col("tr.event_id"), "left")
    .join(event_costs.alias("ec"),
          F.col("e.event_id") == F.col("ec.event_id"), "left")
    .select(
        F.col("e.event_id"),
        F.col("e.org_id"),
        F.col("e.title").alias("event_title"),
        F.col("e.status"),
        F.col("e.start_date"),
        F.col("e.venue_city"),
        F.col("e.venue_state"),
        F.col("e.max_attendees"),
        F.col("tr.total_orders"),
        F.col("tr.gross_ticket_revenue_cents"),
        (F.col("tr.gross_ticket_revenue_cents") / 100.0).alias("gross_ticket_revenue_brl"),
        F.col("tr.avg_ticket_price_cents"),
        F.col("ec.total_cost_cents"),
        # ROI = (receita - custo) / custo
        F.when(F.col("ec.total_cost_cents") > 0,
            (F.col("tr.gross_ticket_revenue_cents") - F.col("ec.total_cost_cents"))
            / F.col("ec.total_cost_cents")
        ).alias("roi_ratio"),
        # Ocupação = pedidos confirmados / capacidade máxima
        F.when(F.col("e.max_attendees") > 0,
            F.col("tr.total_orders") / F.col("e.max_attendees")
        ).alias("occupancy_rate"),
    )
)

(fact_event_revenue.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_bi.fact_event_revenue"))